In [11]:
import torch
import torch.nn as nn
import torch.onnx as onnx

In [12]:
model_path = "simple_linear_model.onnx"

In [13]:
# Create a simple one layer model using a linear layer
class SimpleLinearModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(SimpleLinearModel, self).__init__()
        # Define a single linear layer
        self.linear = nn.Linear(input_size, 10)
        self.linear2 = nn.Linear(10, 20)
        self.linear3 = nn.Linear(20, 15)
        self.linear4 = nn.Linear(15, output_size)

    def forward(self, x):
        # Pass input through the linear layer
        output = self.linear(x)
        output = self.linear2(output)
        output = self.linear3(output)
        output = self.linear4(output)
        return output


# Example usage
model = SimpleLinearModel(input_size=10, output_size=5)
print(model)

print("Model weights:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    if "weight" in name:
        print(f"  Weight values (first 5): {param.flatten()[:5]}")
    elif "bias" in name:
        print(f"  Bias values (first 5): {param.flatten()[:5]}")

SimpleLinearModel(
  (linear): Linear(in_features=10, out_features=10, bias=True)
  (linear2): Linear(in_features=10, out_features=20, bias=True)
  (linear3): Linear(in_features=20, out_features=15, bias=True)
  (linear4): Linear(in_features=15, out_features=5, bias=True)
)
Model weights:
linear.weight: torch.Size([10, 10])
  Weight values (first 5): tensor([ 0.0738, -0.0158,  0.2975, -0.2172, -0.2920], grad_fn=<SliceBackward0>)
linear.bias: torch.Size([10])
  Bias values (first 5): tensor([ 0.3008,  0.3077,  0.0035, -0.2170,  0.1873], grad_fn=<SliceBackward0>)
linear2.weight: torch.Size([20, 10])
  Weight values (first 5): tensor([-0.1330, -0.0287,  0.0763,  0.0134, -0.1325], grad_fn=<SliceBackward0>)
linear2.bias: torch.Size([20])
  Bias values (first 5): tensor([-0.0196,  0.0911,  0.2363,  0.0347, -0.2597], grad_fn=<SliceBackward0>)
linear3.weight: torch.Size([15, 20])
  Weight values (first 5): tensor([ 0.0716,  0.1365,  0.1875, -0.0437,  0.1342], grad_fn=<SliceBackward0>)
linear3.

In [14]:
# export to onnx
onnx.export(model, torch.randn(1, 10), model_path, export_params=True, opset_version=11)

/tmp/ipykernel_18481/837427141.py:2: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  onnx.export(model, torch.randn(1, 10), model_path, export_params=True, opset_version=11)
W0724 17:12:27.982000 18481 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SimpleLinearModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleLinearModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/david/miniconda3/envs/pytorch/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/david/miniconda3/envs/pytorch/lib/python3.14/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/home/david/miniconda3/envs/pytorch/lib/python3.14/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/david/miniconda3/envs/pytor

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.13.0+cu132',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"x"<FLOAT,[1,10]>
            ),
            outputs=(
                %"linear_3"<FLOAT,[1,5]>
            ),
            initializers=(
                %"linear.weight"<FLOAT,[10,10]>{TorchTensor(...)},
                %"linear.bias"<FLOAT,[10]>{TorchTensor<FLOAT,[10]>(Parameter containing: tensor([ 0.3008,  0.3077,  0.0035, -0.2170,  0.1873, -0.0388,  0.1894, -0.1000, -0.0203,  0.0806], requires_grad=True), name='linear.bias')},
                %"linear2.weight"<FLOAT,[20,10]>{TorchTensor(...)},
                %"linear2.bias"<FLOAT,[20]>{TorchTensor(...)},
                %"linear3.weight"<FLOAT,[15,20]>{TorchTensor(...)},
                %"linear3.bias"<FLOAT

In [15]:
# run the model with pytorch
input_data = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]])
with torch.no_grad():
    output = model(input_data)
print(output)

tensor([[-1.3411, -0.1197, -0.5638, -0.3998,  0.7190]])


In [16]:
from c_exporter.onnx_exporter import export_onnx

export_onnx(model_path, "src/model_weights.h")